# 01a · GasWindTunnel: R1 raw features

**Purpose.** Downsample each sensor curve to T' = 100 and flatten (R1). Fit the 4 heads on both tasks with StratifiedKFold(5) stratified by gas × wind speed, 3 seeds.

**Inputs**
- GasWindTunnel `X_signals.npy` (600, 72, 1500) + `meta.csv`

**Outputs**
- F_raw: (600, 72, 100) → (600, 7200)
- `results/01a_gwt_raw.csv`: one row per seed, fold, task, rep, head, metrics

In [1]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.raw import raw_features
DATASET = "gwt"
REP = "R1_raw"
NAME = "01a_gwt_raw"          # results/01a_gwt_raw.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

(600, 72, 1500) [60, 60, 60, 60, 60, 60, 60, 60, 60, 60]


## R1: downsample every curve to T' = 100 points and flatten
Nothing is fitted, so the features are computed once for all samples.

In [2]:
T_PRIME = 100
F = raw_features(X, T_PRIME)               # (N, C * T') = (600, 7200)
assert F.shape == (600, 7200)
F.shape

(600, 7200)

## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters) sees the training fold only.

In [3]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(F, meta, DATASET, task, REP, precomputed)
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

gwt multi R1_raw seed 0 fold 0: logreg 0.967  svm 0.958  rf 0.983  mlp 0.950


gwt multi R1_raw seed 0 fold 1: logreg 0.992  svm 0.983  rf 0.992  mlp 0.975


gwt multi R1_raw seed 0 fold 2: logreg 1.000  svm 0.984  rf 1.000  mlp 0.983


gwt multi R1_raw seed 0 fold 3: logreg 0.975  svm 0.958  rf 0.992  mlp 0.958


gwt multi R1_raw seed 0 fold 4: logreg 0.964  svm 0.975  rf 0.983  mlp 0.958


gwt multi R1_raw seed 1 fold 0: logreg 0.975  svm 0.959  rf 1.000  mlp 0.950


... (20 outputs left out)

gwt binary R1_raw seed 2 fold 1: logreg 0.976  svm 0.950  rf 1.000  mlp 0.976


gwt binary R1_raw seed 2 fold 2: logreg 1.000  svm 0.976  rf 1.000  mlp 1.000


gwt binary R1_raw seed 2 fold 3: logreg 1.000  svm 1.000  rf 1.000  mlp 0.976


gwt binary R1_raw seed 2 fold 4: logreg 0.957  svm 0.954  rf 1.000  mlp 0.954
saved results/01a_gwt_raw.csv


## Results: mean ± std over 3 seeds × 5 folds

In [4]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

macro F1


head                          logreg            mlp             rf  \
dataset task   rep                                                   
gwt     binary R1_raw  0.982 ± 0.022  0.974 ± 0.032  1.000 ± 0.000   
        multi  R1_raw  0.980 ± 0.012  0.956 ± 0.018  0.992 ± 0.008   

head                             svm  
dataset task   rep                    
gwt     binary R1_raw  0.972 ± 0.032  
        multi  R1_raw  0.968 ± 0.016

balanced accuracy


head                          logreg            mlp             rf  \
dataset task   rep                                                   
gwt     binary R1_raw  0.984 ± 0.026  0.965 ± 0.048  1.000 ± 0.000   
        multi  R1_raw  0.980 ± 0.011  0.956 ± 0.018  0.992 ± 0.008   

head                             svm  
dataset task   rep                    
gwt     binary R1_raw  0.963 ± 0.047  
        multi  R1_raw  0.968 ± 0.016

In [5]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")

ROC AUC, binary task


,,head,logreg,mlp,rf,svm
dataset,task,rep,,,,
gwt,binary,R1_raw,0.996 ± 0.007,0.989 ± 0.019,1.000 ± 0.000,0.980 ± 0.038
